# Feature Engineering — Campus Waste Intelligence System

## 0. Setup

In [3]:
import os
import pandas as pd
import numpy as np

# # ── Colab / Drive mount (comment out if running locally) ──────────────────────
# try:
#     from google.colab import drive
#     drive.mount('/content/drive', force_remount=True)
#     os.chdir("/content/drive/MyDrive/UAB/FDS/campus-waste-intelligence")
#     print("Drive mounted and directory set.")
# except Exception:
#     print("Not in Colab — using local paths.")

## 1. Load Preprocessed Data

In [4]:
df_raw = pd.read_csv('data/food_waste_preprocessed.csv')
df_raw['time_bin'] = pd.to_datetime(df_raw['time_bin'])

print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw['time_bin'].min()}  →  {df_raw['time_bin'].max()}")
print(f"Sections: {sorted(df_raw['Canteen_Section'].unique())}")
df_raw[['Canteen_Section', 'time_bin', 'Waste_Weight_kg', 'Foot_Traffic', 'is_weekend']].head()

Shape: (74413, 20)
Date range: 2015-01-01 06:00:00  →  2025-08-10 19:30:00
Sections: ['A', 'B', 'C', 'D']


,Canteen_Section,time_bin,Waste_Weight_kg,Foot_Traffic,is_weekend
0,A,2015-01-01 12:00:00,2.98,120.0,0
1,A,2015-01-01 17:30:00,1.15,100.0,0
2,A,2015-01-01 18:00:00,0.93,100.0,0
3,A,2015-01-02 06:00:00,0.44,80.0,0
4,A,2015-01-02 12:00:00,1.54,120.0,0


## 2. Holiday Flags
Replicates the work from the previous notebook so this notebook is self-contained.

In [5]:
df = df_raw.copy()

try:
    import holidays
    years = df['time_bin'].dt.year.unique().tolist()
    us_holidays = holidays.USA(years=years)
    df['is_holiday'] = df['time_bin'].dt.date.apply(
        lambda d: 1 if d in us_holidays else 0
    )
except ImportError:
    print("Install 'holidays' package: pip install holidays")
    df['is_holiday'] = 0

df['is_special_day'] = ((df['is_holiday'] == 1) | (df['is_weekend'] == 1)).astype(int)
print(f"Holiday observations : {df['is_holiday'].sum()}")
print(f"Special-day observations: {df['is_special_day'].sum()}")

Holiday observations : 2433
Special-day observations: 22955


## 3. Aggregate to Daily Level

All four models work on a **daily** granularity:
- Sub-daily observations are too noisy for a 7-day-ahead forecast.
- Aggregating per `(date, Canteen_Section)` gives us the natural unit of prediction.

We sum waste and cost, take the daily mean foot traffic, and preserve flag columns.

In [6]:
df['date'] = df['time_bin'].dt.date

daily = (
    df.groupby(['date', 'Canteen_Section'])
    .agg(
        Waste_Weight_kg = ('Waste_Weight_kg', 'sum'),   # total daily waste
        Cost_Loss       = ('Cost_Loss',       'sum'),   # total daily cost
        Foot_Traffic    = ('Foot_Traffic',    'mean'),  # avg daily traffic
        is_holiday      = ('is_holiday',      'max'),   # 1 if any bin was a holiday
        is_special_day  = ('is_special_day',  'max'),
        is_weekend      = ('is_weekend',      'max'),
    )
    .reset_index()
)

daily['date'] = pd.to_datetime(daily['date'])
daily = daily.sort_values(['Canteen_Section', 'date']).reset_index(drop=True)

print(f"Daily dataset shape: {daily.shape}")
daily.head(10)

Daily dataset shape: (15431, 8)


,date,Canteen_Section,Waste_Weight_kg,Cost_Loss,Foot_Traffic,is_holiday,is_special_day,is_weekend
0,2015-01-01,A,5.06,12.18,106.666667,1,1,0
1,2015-01-02,A,8.20,31.50,105.000000,0,0,0
2,2015-01-03,A,18.35,68.74,70.000000,0,1,1
3,2015-01-04,A,3.72,29.77,70.000000,0,1,1
4,2015-01-05,A,22.08,83.26,88.571429,0,0,0
5,2015-01-06,A,25.28,46.62,102.222222,0,0,0
6,2015-01-07,A,3.91,22.96,106.666667,0,0,0
7,2015-01-08,A,2.70,6.14,80.000000,0,0,0
8,2015-01-09,A,22.05,122.78,91.428571,0,0,0
9,2015-01-10,A,15.29,71.57,65.333333,0,1,1


## 4. Calendar Features
Clean, interpretable calendar features derived from the date.  
These are used directly by **XGBoost** and as optional regressors/exogenous variables for **Prophet** and **SARIMA**.

In [7]:
daily['weekday']     = daily['date'].dt.dayofweek          # 0=Mon … 6=Sun
daily['month']       = daily['date'].dt.month
daily['quarter']     = daily['date'].dt.quarter
daily['day_of_year'] = daily['date'].dt.dayofyear
daily['week_of_year']= daily['date'].dt.isocalendar().week.astype(int)
daily['year']        = daily['date'].dt.year

# ── Cyclical encodings (XGBoost needs these; Prophet/SARIMA don't) ────────────
# Encodes cyclic structure so weekday 6 → weekday 0 is continuous, not a jump.
daily['sin_weekday'] = np.sin(2 * np.pi * daily['weekday'] / 7)
daily['cos_weekday'] = np.cos(2 * np.pi * daily['weekday'] / 7)
daily['sin_month']   = np.sin(2 * np.pi * daily['month']   / 12)
daily['cos_month']   = np.cos(2 * np.pi * daily['month']   / 12)
daily['sin_doy']     = np.sin(2 * np.pi * daily['day_of_year'] / 365)
daily['cos_doy']     = np.cos(2 * np.pi * daily['day_of_year'] / 365)

print("Calendar features added.")
daily[['date', 'weekday', 'month', 'sin_weekday', 'cos_weekday']].head()

Calendar features added.


,date,weekday,month,sin_weekday,cos_weekday
0,2015-01-01,3,1,0.433884,-0.900969
1,2015-01-02,4,1,-0.433884,-0.900969
2,2015-01-03,5,1,-0.974928,-0.222521
3,2015-01-04,6,1,-0.781831,0.623490
4,2015-01-05,0,1,0.000000,1.000000


## 5. Lag Features  *(XGBoost only)*

Lags give the model direct access to recent history without requiring internal state.  
We compute lags **within each section** so section A's history never contaminates section B.

| Lag | Reasoning |
|---|---|
| `lag_1` | Yesterday's waste — strongest short-term signal |
| `lag_7` | Same weekday last week — captures weekly seasonality |
| `lag_14` | Two weeks ago — cross-checks the weekly pattern |
| `lag_28` | Four weeks ago — accounts for monthly rhythms |

In [8]:
for lag in [1, 7, 14, 28]:
    daily[f'lag_{lag}'] = (
        daily.groupby('Canteen_Section')['Waste_Weight_kg']
        .shift(lag)
    )

print("Lag features added.")
daily[['date', 'Canteen_Section', 'Waste_Weight_kg',
       'lag_1', 'lag_7', 'lag_14', 'lag_28']].head(35).tail(10)

Lag features added.


,date,Canteen_Section,Waste_Weight_kg,lag_1,lag_7,lag_14,lag_28
25,2015-01-26,A,19.96,9.73,18.45,24.17,NaN
26,2015-01-27,A,11.72,19.96,4.74,14.21,NaN
27,2015-01-28,A,15.06,11.72,20.69,11.71,NaN
28,2015-01-29,A,10.76,15.06,8.79,7.85,5.06
29,2015-01-30,A,14.24,10.76,24.92,22.49,8.20
30,2015-01-31,A,18.52,14.24,12.58,9.37,18.35
31,2015-02-01,A,6.50,18.52,9.73,5.62,3.72
32,2015-02-02,A,20.55,6.50,19.96,18.45,22.08
33,2015-02-03,A,3.25,20.55,11.72,4.74,25.28
34,2015-02-04,A,21.08,3.25,15.06,20.69,3.91


## 6. Rolling Window Statistics  *(XGBoost only)*

Rolling features smooth short-term noise and encode recent trend/volatility.  
Windows are computed on a **min_periods=1** basis so the first rows are not all NaN.

| Feature | What it captures |
|---|---|
| `rolling_mean_7` | Recent 7-day average — short-term trend |
| `rolling_mean_14` | 14-day average — medium-term trend |
| `rolling_std_7` | 7-day standard deviation — volatility signal |
| `rolling_max_7` | 7-day peak — captures spike risk |

In [9]:
def rolling_group(df, col, window, func):
    """Apply rolling function within each Canteen_Section group."""
    return (
        df.groupby('Canteen_Section')[col]
        .transform(lambda x: getattr(x.shift(1).rolling(window, min_periods=1), func)())
        # shift(1) so we never leak today's value into today's features
    )

daily['rolling_mean_7']  = rolling_group(daily, 'Waste_Weight_kg', 7,  'mean')
daily['rolling_mean_14'] = rolling_group(daily, 'Waste_Weight_kg', 14, 'mean')
daily['rolling_std_7']   = rolling_group(daily, 'Waste_Weight_kg', 7,  'std')
daily['rolling_max_7']   = rolling_group(daily, 'Waste_Weight_kg', 7,  'max')

# Fill any remaining NaNs in std (only on very first row per section)
daily['rolling_std_7'].fillna(0, inplace=True)

print("Rolling features added.")
daily[['date', 'Canteen_Section', 'Waste_Weight_kg',
       'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'rolling_max_7']].head(15)

Rolling features added.


C:\Users\hp\AppData\Local\Temp\ipykernel_22024\2842289347.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  daily['rolling_std_7'].fillna(0, inplace=True)


,date,Canteen_Section,Waste_Weight_kg,rolling_mean_7,rolling_mean_14,rolling_std_7,rolling_max_7
0,2015-01-01,A,5.06,NaN,NaN,NaN,NaN
1,2015-01-02,A,8.20,5.060000,5.060000,NaN,5.06
2,2015-01-03,A,18.35,6.630000,6.630000,2.220315,8.20
3,2015-01-04,A,3.72,10.536667,10.536667,6.946296,18.35
4,2015-01-05,A,22.08,8.832500,8.832500,6.616955,18.35
5,2015-01-06,A,25.28,11.482000,11.482000,8.242410,22.08
6,2015-01-07,A,3.91,13.781667,13.781667,9.277966,25.28
7,2015-01-08,A,2.70,12.371429,12.371429,9.255014,25.28
8,2015-01-09,A,22.05,12.034286,11.162500,9.602218,25.28
9,2015-01-10,A,15.29,14.012857,12.372222,10.094761,25.28


## 7. Location Encoding  *(XGBoost only)*

XGBoost needs numeric input. We label-encode `Canteen_Section` (A=0, B=1, …)  
and also create one-hot columns for interpretability.

In [10]:
section_map = {s: i for i, s in enumerate(sorted(daily['Canteen_Section'].unique()))}
daily['section_code'] = daily['Canteen_Section'].map(section_map)

# One-hot columns (drop_first avoids multicollinearity)
ohe = pd.get_dummies(daily['Canteen_Section'], prefix='section', drop_first=True)
daily = pd.concat([daily, ohe], axis=1)

print(f"Section mapping: {section_map}")
print(f"OHE columns added: {list(ohe.columns)}")

Section mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3}
OHE columns added: ['section_B', 'section_C', 'section_D']


## 8. Handle NaNs from Lag/Rolling

The first `lag_28` rows per section will be NaN. We drop them from the **XGBoost dataset**  
but keep the full daily frame for Prophet and SARIMA.

In [11]:
lag_cols = ['lag_1', 'lag_7', 'lag_14', 'lag_28']
nan_before = daily[lag_cols].isna().sum()
print("NaNs per lag column before dropping:")
print(nan_before)

# Rows to drop: any section row where lag_28 is NaN (first 28 rows of each section)
rows_to_drop = daily[lag_cols].isna().any(axis=1).sum()
print(f"\nRows with any lag NaN: {rows_to_drop} ({rows_to_drop/len(daily)*100:.1f}%)")

NaNs per lag column before dropping:
lag_1       4
lag_7      28
lag_14     56
lag_28    112
dtype: int64

Rows with any lag NaN: 112 (0.7%)


## 9. Final Datasets

We export **three** CSVs:

| File | Used by | Notes |
|---|---|---|
| `waste_features_full.csv` | Prophet, SARIMA, MA | All daily rows, all sections |
| `waste_features_xgb.csv` | XGBoost | Drops lag-NaN rows; includes all engineered features |
| `waste_features_per_section/` | SARIMA (one series per section) | Optional — separate series |

In [12]:
import os
os.makedirs('data', exist_ok=True)

# ── 9a. Full daily dataset (Prophet / SARIMA / Moving Average) ────────────────
# Prophet needs: ds, y  +  optional regressors
# SARIMA needs:  a clean index + optional exog columns
# MA needs:      just date + Waste_Weight_kg

prophet_sarima_cols = [
    'date',             # rename to 'ds' when feeding Prophet
    'Canteen_Section',
    'Waste_Weight_kg',  # rename to 'y' when feeding Prophet
    'Foot_Traffic',     # exogenous / regressor
    'is_holiday',       # exogenous / regressor
    'is_special_day',   # exogenous / regressor
    'is_weekend',       # exogenous / regressor
    'weekday',
    'month',
    'week_of_year',
]

df_full = daily[prophet_sarima_cols].copy()
df_full.to_csv('data/waste_features_full.csv', index=False)
print(f"[waste_features_full.csv]  shape: {df_full.shape}")
df_full.head()

[waste_features_full.csv]  shape: (15431, 10)


,date,Canteen_Section,Waste_Weight_kg,Foot_Traffic,is_holiday,is_special_day,is_weekend,weekday,month,week_of_year
0,2015-01-01,A,5.06,106.666667,1,1,0,3,1,1
1,2015-01-02,A,8.20,105.000000,0,0,0,4,1,1
2,2015-01-03,A,18.35,70.000000,0,1,1,5,1,1
3,2015-01-04,A,3.72,70.000000,0,1,1,6,1,1
4,2015-01-05,A,22.08,88.571429,0,0,0,0,1,2


In [13]:
# ── 9b. XGBoost dataset (drop NaN lag rows) ───────────────────────────────────

xgb_cols = [
    'date',
    'Canteen_Section',
    'section_code',       # numeric label encoding
    'Waste_Weight_kg',    # TARGET
    # --- Exogenous / context ---
    'Foot_Traffic',
    'is_holiday',
    'is_special_day',
    'is_weekend',
    # --- Calendar (numeric) ---
    'weekday',
    'month',
    'quarter',
    'day_of_year',
    'week_of_year',
    'year',
    # --- Cyclical ---
    'sin_weekday', 'cos_weekday',
    'sin_month',   'cos_month',
    'sin_doy',     'cos_doy',
    # --- Lags ---
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    # --- Rolling stats ---
    'rolling_mean_7', 'rolling_mean_14',
    'rolling_std_7',  'rolling_max_7',
] + [c for c in daily.columns if c.startswith('section_') and c != 'section_code']

df_xgb = daily[xgb_cols].dropna(subset=lag_cols).reset_index(drop=True)
df_xgb.to_csv('data/waste_features_xgb.csv', index=False)
print(f"[waste_features_xgb.csv]   shape: {df_xgb.shape}")
df_xgb.head()

[waste_features_xgb.csv]   shape: (15319, 31)


,date,Canteen_Section,section_code,Waste_Weight_kg,Foot_Traffic,is_holiday,is_special_day,is_weekend,weekday,month,...,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_14,rolling_std_7,rolling_max_7,section_B,section_C,section_D
0,2015-01-29,A,0,10.76,104.0,0,0,0,3,1,...,8.79,7.85,5.06,14.680000,13.712143,5.848741,24.92,False,False,False
1,2015-01-30,A,0,14.24,97.5,0,0,0,4,1,...,24.92,22.49,8.20,14.961429,13.920000,5.558274,24.92,False,False,False
2,2015-01-31,A,0,18.52,68.0,0,1,1,5,1,...,12.58,9.37,18.35,13.435714,13.330714,3.425862,19.96,False,False,False
3,2015-02-01,A,0,6.50,70.0,0,1,1,6,2,...,9.73,5.62,3.72,14.284286,13.984286,3.883649,19.96,False,False,False
4,2015-02-02,A,0,20.55,194.4,0,0,0,0,2,...,19.96,18.45,22.08,13.822857,14.047143,4.634284,19.96,False,False,False


In [14]:
# ── 9c. Per-section CSVs (convenient for SARIMA — one univariate series each) ─
os.makedirs('data/per_section', exist_ok=True)

for section, grp in df_full.groupby('Canteen_Section'):
    path = f'data/per_section/waste_section_{section}.csv'
    grp.sort_values('date').to_csv(path, index=False)
    print(f"  Saved {path}  ({len(grp)} rows)")

  Saved data/per_section/waste_section_A.csv  (3848 rows)
  Saved data/per_section/waste_section_B.csv  (3859 rows)
  Saved data/per_section/waste_section_C.csv  (3861 rows)
  Saved data/per_section/waste_section_D.csv  (3863 rows)


## 10. Feature Summary & Model Mapping

### Moving Average Baseline
```python
# Load waste_features_full.csv, filter one section, use Waste_Weight_kg only.
# Compute 7-day rolling mean as the forecast.
```
**Columns needed**: `date`, `Waste_Weight_kg`

---

### SARIMA / SARIMAX
```python
# Load waste_features_full.csv, filter one section (or total across sections).
# Set date as index. Use Waste_Weight_kg as endog.
# Optional exog: ['Foot_Traffic', 'is_holiday', 'is_special_day']
```
**Columns needed**: `date`, `Waste_Weight_kg`, optional exog above  
**File**: `waste_features_full.csv` or `per_section/waste_section_*.csv`

---

### Prophet
```python
# Load waste_features_full.csv, filter one section.
# Rename: date → ds, Waste_Weight_kg → y
# Add regressors: Foot_Traffic, is_holiday, is_special_day
# Prophet handles weekly + yearly seasonality internally — don't add those manually.
```
**Columns needed**: `ds (date)`, `y (Waste_Weight_kg)`, optional regressors above  
**File**: `waste_features_full.csv`

---

### XGBoost
```python
# Load waste_features_xgb.csv.
# TARGET  : Waste_Weight_kg
# FEATURES: everything else except 'date' and 'Canteen_Section' (string)
# Train/test split: temporal — last N days as test (DO NOT shuffle)
# For 7-day-ahead prediction: use lag_7 as the minimum look-back feature.
```
**Columns needed**: all columns except `date`, `Canteen_Section`  
**File**: `waste_features_xgb.csv`


## 11. Quick Sanity Check

In [15]:
print("=" * 55)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 55)
print(f"\nOriginal sub-daily observations : {len(df_raw):,}")
print(f"Daily aggregated rows (all sections): {len(daily):,}")
print(f"  waste_features_full.csv   : {df_full.shape}")
print(f"  waste_features_xgb.csv    : {df_xgb.shape}  (lag NaNs dropped)")

print("\n--- XGBoost feature columns ---")
feature_cols = [c for c in df_xgb.columns
                if c not in ('date', 'Canteen_Section', 'Waste_Weight_kg')]
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

print("\n--- NaN check (XGBoost df) ---")
print(df_xgb.isna().sum()[df_xgb.isna().sum() > 0])
if df_xgb.isna().sum().sum() == 0:
    print("  ✅ No NaNs in XGBoost dataset.")

print("\n--- Daily waste stats (all sections combined) ---")
print(df_xgb['Waste_Weight_kg'].describe().round(2))

FEATURE ENGINEERING SUMMARY

Original sub-daily observations : 74,413
Daily aggregated rows (all sections): 15,431
  waste_features_full.csv   : (15431, 10)
  waste_features_xgb.csv    : (15319, 31)  (lag NaNs dropped)

--- XGBoost feature columns ---
   1. section_code
   2. Foot_Traffic
   3. is_holiday
   4. is_special_day
   5. is_weekend
   6. weekday
   7. month
   8. quarter
   9. day_of_year
  10. week_of_year
  11. year
  12. sin_weekday
  13. cos_weekday
  14. sin_month
  15. cos_month
  16. sin_doy
  17. cos_doy
  18. lag_1
  19. lag_7
  20. lag_14
  21. lag_28
  22. rolling_mean_7
  23. rolling_mean_14
  24. rolling_std_7
  25. rolling_max_7
  26. section_B
  27. section_C
  28. section_D

--- NaN check (XGBoost df) ---
Series([], dtype: int64)
  ✅ No NaNs in XGBoost dataset.

--- Daily waste stats (all sections combined) ---
count    15319.00
mean        11.98
std          5.78
min          0.17
25%          7.79
50%         11.47
75%         15.62
max         43.76
Name: 